# Capstone #1 — Research Agent

*Notebook 13*

Put it all together. Combine web search, file search, and code interpreter into one agent that researches a topic, analyzes documents, and produces a structured report.
<br>
<br>

**Topics:**
- Combining multiple built-in tools in one agent

- Web search + file search + code interpreter pipeline

- Structured report generation with Pydantic

- Wrapping the run in try/except for robustness

- Evaluating the agent against a golden test set

---

## 🔧 Setup

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
load_dotenv(dotenv_path=Path("..") / ".env")

from openai import OpenAI
from agents import Agent, CodeInterpreterTool, FileSearchTool, Runner, WebSearchTool

MODEL = "gpt-5-mini"
REASONING_MODEL = "gpt-5"

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

print(f"✅ Setup complete: Using {MODEL}")

---

## 🏗️ System Architecture

The research agent combines all three built-in tools from Lessons 10–12:
- **Web Search** (Lesson 10) — finds current information and news
- **File Search** (Lesson 11) — queries private reference documents
- **Code Interpreter** (Lesson 12) — computes and analyzes quantitative data

**Workflow:** Topic → agent autonomously chooses tools → structured report. (Capstone 2 will introduce explicit multi-agent pipelines you orchestrate phase-by-phase.)

---

## 📄 Phase 1: Build the Knowledge Base

We'll create an internal reference document and upload it to a vector store (an OpenAI-managed collection that indexes your files for retrieval by `FileSearchTool`) — giving the agent access to private context that web search won't find.

In [ ]:
# Internal reference document — private context the web doesn't have
reference_doc = """# AI Adoption Survey — Internal Research Summary

## Survey Overview
Conducted Q3 of this year. 847 respondents across technology, healthcare,
finance, and retail sectors. Companies ranged from 50 to 10,000+ employees.

## Key Findings

### Adoption Rates
- 73% of companies surveyed are actively using AI tools in production
- 18% are in pilot or evaluation phase
- 9% have no AI initiatives currently

### Primary Use Cases (ranked by adoption)
1. Customer support automation — 61%
2. Data analysis and reporting — 58%
3. Content generation — 47%
4. Code assistance — 44%
5. Document processing — 39%

### Biggest Barriers to Adoption
- Data privacy and security concerns — 67%
- Integration with existing systems — 54%
- Lack of skilled staff — 48%
- Cost and ROI uncertainty — 43%

### Budget Trends
- Average AI budget increase year-over-year: 34%
- Companies with dedicated AI teams: 41%
- Expected to increase AI investment next year: 78%

### Satisfaction
- Exceeded expectations: 29%
- Met expectations: 51%
- Below expectations: 20%
"""

# Save locally
doc_path = Path("ai_adoption_survey.txt")
doc_path.write_text(reference_doc)

# --------------------------------------------------------------
print(f"✅ Reference document created: {doc_path}")

#### Upload to Vector Store

In [ ]:
print("Uploading to vector store...\n")

vector_store = client.vector_stores.create(name="Research Agent Knowledge Base")

with open(doc_path, "rb") as file_handle:
    file_batch = client.vector_stores.file_batches.upload_and_poll(
        vector_store_id=vector_store.id,
        files=[file_handle]
    )

# --------------------------------------------------------------
print(f"✅ Vector store ready: {vector_store.id}")
print(f"   Files: {file_batch.file_counts.completed} processed")

With the private knowledge base ready, we can now build a single agent that has access to both public web data and our internal survey.

## 🤖 Phase 2: Build the Research Agent

All three tools in one agent. The instructions tell it exactly how to use each one.

Pattern to copy: list every tool the agent might need, give it numbered step-by-step instructions for when to use each, and start with default parameters. In your own project, you'd swap three things — the vector store contents, the agent instructions, and the `ResearchReport` fields.

In [ ]:
from pydantic import BaseModel
from typing import List

class ResearchReport(BaseModel):
    executive_summary: str
    key_findings: List[str]
    data_analysis: str
    sources: List[str]

research_instructions = (
    "You are a thorough research analyst. When given a research topic:\n\n"
    "1. Use web search to find current information, news, and trends\n"
    "2. Use file search to find relevant internal data and context\n"
    "3. Use code interpreter to compute statistics or analyze any numerical data\n"
    "4. Synthesize all findings into a structured report with these sections:\n"
    "    - Executive Summary (2-3 sentences)\n"
    "    - Key Findings (bullet points)\n"
    "    - Data Analysis (numbers and statistics)\n"
    "    - Sources\n\n"
    "Always cite your sources. Be specific with numbers."
)

research_agent = Agent(
    name="ResearchAgent",
    instructions=research_instructions,
    model=MODEL,
    output_type=ResearchReport,
    tools=[
        WebSearchTool(),
        FileSearchTool(
            vector_store_ids=[vector_store.id],
            max_num_results=3
        ),
        CodeInterpreterTool(tool_config={
            "type": "code_interpreter",
            "container": {"type": "auto"}  # "auto" lets the SDK reuse a sandbox across calls — faster and cheaper
        })
    ]
)

# --------------------------------------------------------------
print("✅ Research agent ready")
print("    Tools: WebSearch + FileSearch + CodeInterpreter")

⚠️ **Security note:** Web search results and retrieved document chunks are untrusted input (see Lesson 23: Prompt Injection & Tool Safety). In production, validate or sanitize tool output before passing it into system instructions or downstream tools.

---

In [ ]:
research_topic = "Current state of AI adoption in enterprise — trends, barriers, and outlook"

print("🔬 RESEARCH AGENT RUNNING")
print("=" * 60)
print(f"Topic: {research_topic}")
print("\nResearching... (this may take 20-30 seconds)\n")

# Robust version — wrap multi-tool runs in try/except; failures can happen at any tool step
try:
    research_result = await Runner.run(research_agent, input=research_topic)
except Exception as e:
    print(f"Research failed: {e}")
    research_result = None

if research_result:
    report = research_result.final_output

    print("=" * 60)
    print("📋 RESEARCH REPORT")
    print("=" * 60)
    print(f"\nExecutive Summary:\n{report.executive_summary}\n")
    print("Key Findings:")
    for finding in report.key_findings:
        print(f"  • {finding}")
    print(f"\nData Analysis:\n{report.data_analysis}\n")
    print("Sources:")
    for source in report.sources:
        print(f"  • {source}")
    print("=" * 60)

    # Because we used output_type=ResearchReport, `report` is a validated Python object
    # with .executive_summary, .key_findings, etc. — safe to pass directly to other code,
    # databases, or downstream agents.

    print(f"\nTools called: {[item.type for item in research_result.new_items if hasattr(item, 'type') and item.type.endswith('_call')]}")

In [ ]:
# Clean up local sample files before exercises
for file in ["ai_adoption_survey.txt"]:
    path = Path(file)
    if path.exists():
        path.unlink()
        print(f"✅ Local file deleted: {file}")

---

## 💪 Practice Exercises

### Exercise 1: Export Research Report to Markdown

*Covers: saving agent output — writing results to a file*

Save the research report to a `.md` file using the existing `research_result`.

In [ ]:
# --------------------------------------------------------------
# 💪 Exercise 1: Export Research Report to Markdown
# --------------------------------------------------------------
# Objective: Save the existing research report to a markdown file.

from datetime import datetime

# TODO 1: Create a markdown header with the topic and current date
#          Hint: date_str = datetime.now().strftime("%Y-%m-%d")
#                header = f"# Research Report: {research_topic}\nDate: {date_str}\n\n"

# TODO 2: Combine the header with research_result.final_output
#          (If output_type=ResearchReport, use research_result.final_output.executive_summary etc.)

# TODO 3: Write the combined content to "report.md" using Path("report.md").write_text()

# TODO 4: Print a confirmation message showing the file path

# --- Write your code below this line ---

---

### Exercise 2: Evaluate Against a Golden Test Set

*Covers: evaluation harness — scoring with a judge agent*

Apply the judge-agent evaluation pattern from Lesson 9 to one research report (focus on the judge step, not building a full test harness). Good golden tests for this kind of agent check both content coverage (did it include required facts?) and tool-backed grounding (did it cite sources or use internal data when expected?).

In [ ]:
# --------------------------------------------------------------
# 💪 Exercise 2: Evaluate Against a Golden Test Set
# --------------------------------------------------------------
# Objective: Score one research report using a judge agent (Lesson 9 pattern).

from pydantic import BaseModel

# A golden test — the topic we already ran, with criteria the report should meet
golden_test = {
    "topic": "Current state of AI adoption in enterprise",
    "must_contain": ["executive summary", "73%"],  # from internal survey
}

# Define an EvalResult model for the judge agent
class EvalResult(BaseModel):
    passed: bool
    score: int  # 0-10
    feedback: str

# TODO 1: Create a judge agent using REASONING_MODEL and output_type=EvalResult
#          Instructions: evaluate whether the report meets the expected criteria

# TODO 2: Run the judge on the existing research report, passing the must_contain criteria

# --- Write your code below this line ---

---

## 🎯 Key Takeaways

**Tools Work Together:**

- One agent can hold multiple tools — it chooses the right one for each step

- Web search for current info, file search for private context, code interpreter for computation
<br>
<br>

**Instructions Drive Tool Use:**

- Explicit instructions about when to use each tool produce more consistent results

- A structured output format (report sections) makes the agent's output predictable
<br>
<br>

**Error Handling and Structured Output:**

- Wrap Runner.run() in try/except for multi-tool agents — failures can happen at any tool step

- Use Pydantic output_type= for machine-readable report structure, not just prompt instructions

---

## 📍 Next Step

**Notebook 14: Handoffs**  

Learn how to route tasks between specialized agents using the handoff pattern.

---

##### 🔧 [Troubleshooting Guide](https://github.com/barrettscott/openai-agents/blob/main/TROUBLESHOOTING.md#lesson-13-capstone-1--research-agent)

---

#### Cleanup: Delete the Vector Store

Vector stores persist on OpenAI's servers until deleted. Run this cell when you're done to avoid ongoing storage charges.

In [ ]:
client.vector_stores.delete(vector_store.id)
print(f"✅ Vector store deleted: {vector_store.id}")

---